# GoogLeNet Training — Brain Tumor Detection

Two variants:
1. **Custom GoogLeNet** — trained from scratch using our `googlenet.py` implementation
2. **Pretrained GoogLeNet** — torchvision's `googlenet` with ImageNet weights, fine-tuned

In [1]:
!rm -rf tumor-detection-roboflow
!git clone -b GoogleNet https://github.com/HamzaElhmd/tumor-detection-roboflow.git
!cd tumor-detection-roboflow

Cloning into 'tumor-detection-roboflow'...
remote: Enumerating objects: 336, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 336 (delta 0), reused 4 (delta 0), pack-reused 329 (from 1)
Receiving objects: 100% (336/336), 104.27 MiB | 17.80 MiB/s, done.
Resolving deltas: 100% (21/21), done.


In [2]:
import torch
import torch.nn as nn
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from pathlib import Path
import sys

ROOT = Path.cwd() / 'tumor-detection-roboflow'
sys.path.insert(0, str(ROOT))
print(ROOT)

from preprocess import ImagePreprocessor
from googlenet import GoogLeNet
import copy
import os


if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

/content/tumor-detection-roboflow
Using device: cuda


In [3]:
train_dir = ROOT / "brain-Tumor-1/train"
test_dir  = ROOT / "brain-Tumor-1/test"
valid_dir = ROOT / "brain-Tumor-1/valid"

preprocessor = ImagePreprocessor(size=(224, 224))

train_dataset = ImageFolder(train_dir, transform=preprocessor.train_transform)
test_dataset  = ImageFolder(test_dir,  transform=preprocessor.test_transform)
valid_dataset = ImageFolder(valid_dir, transform=preprocessor.test_transform)

BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Classes: {train_dataset.classes}")
print(f"Train samples: {len(train_dataset)}")
print(f"Test  samples: {len(test_dataset)}")
print(f"Valid samples: {len(valid_dataset)}")

Classes: ['NoTumor', 'Tumor']
Train samples: 160
Test  samples: 23
Valid samples: 45


## Preprocessed Data Verification

Check the preprocessed pipeline output before training: pixel statistics on normalized data and per-channel distributions.

In [ ]:
from EDA.distribution import preprocessed_pixel_statistics, preprocessed_channel_distributions

# Verify normalized pixel statistics on the training set
train_stats = preprocessed_pixel_statistics(train_loader)
print("Preprocessed training set statistics:")
print(f"  Num images: {train_stats['num_images']}")
print(f"  Shape:      {train_stats['shape']}")
print(f"  Min  (R,G,B): {train_stats['min_per_channel']}")
print(f"  Max  (R,G,B): {train_stats['max_per_channel']}")
print(f"  Mean (R,G,B): {train_stats['mean_per_channel']}")
print(f"  Std  (R,G,B): {train_stats['std_per_channel']}")

# Per-channel pixel value distributions on the validation set
valid_hists = preprocessed_channel_distributions(valid_loader)
print(f"\nPreprocessed validation set histograms (bins={len(valid_hists['bins'])-1})")
for ch in ['R', 'G', 'B']:
    print(f"  {ch}: total pixels counted = {valid_hists[ch].sum():.0f}")

# Quick sanity check: denormalize one batch and inspect the raw first image
from EDA.distribution import load_images_from_dataloader
sample_imgs = load_images_from_dataloader(train_loader, denormalize=True, max_batches=1)
print(f"\nDenormalized sample image range: [{sample_imgs[0].min():.3f}, {sample_imgs[0].max():.3f}]")
print(f"Denormalized sample image shape: {sample_imgs[0].shape}")

## 1. Custom GoogLeNet — Trained from Scratch

In [11]:
model_scratch = GoogLeNet(lr=1e-4, weight_decay=1e-4, dropout_prob=0.4, aux_weight=0.3)
model_scratch.to(device)

total_params = sum(p.numel() for p in model_scratch.parameters() if p.requires_grad)
print(f"Trainable parameters: {total_params:,}")

Trainable parameters: 10,306,355


In [12]:
def train_one_epoch(model, loader, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        model.optimizer.zero_grad()
        outputs = model(images)
        loss = model.compute_loss(outputs, labels.float())
        loss.backward()
        model.optimizer.step()

        running_loss += loss.item() * images.size(0)

        # accuracy from main logits
        logits = outputs[0] if isinstance(outputs, tuple) else outputs
        probs = torch.sigmoid(logits)
        preds = (probs >= 0.5).long().squeeze(1)
        correct += (preds == labels.long().to(device)).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total


def evaluate(model, loader, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            logits = outputs[0] if isinstance(outputs, tuple) else outputs
            loss = model.criterion(logits, labels.float().view(-1, 1))

            running_loss += loss.item() * images.size(0)
            probs = torch.sigmoid(logits)
            preds = (probs >= 0.5).long().squeeze(1)
            correct += (preds == labels.long().to(device)).sum().item()
            total += labels.size(0)

    return running_loss / total, correct / total

In [15]:
EPOCHS = 30
best_acc = 0.0
best_weights = None

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(model_scratch, train_loader, device)
    valid_loss, valid_acc = evaluate(model_scratch, valid_loader, device)

    if valid_acc > best_acc:
        best_acc = valid_acc
        best_weights = copy.deepcopy(model_scratch.state_dict())

    print(f"Epoch {epoch:2d}/{EPOCHS} | "
          f"Train Loss: {train_loss:.4f}  Acc: {train_acc:.4f} | "
          f"Valid Loss: {valid_loss:.4f}  Acc: {valid_acc:.4f}")

print(f"\nBest validation accuracy: {best_acc:.4f}")

Epoch  1/30 | Train Loss: 0.5623  Acc: 0.8500 | Valid Loss: 0.6387  Acc: 0.7778
Epoch  2/30 | Train Loss: 0.6544  Acc: 0.8562 | Valid Loss: 0.5595  Acc: 0.7556
Epoch  3/30 | Train Loss: 0.5275  Acc: 0.8688 | Valid Loss: 0.4771  Acc: 0.8000
Epoch  4/30 | Train Loss: 0.5930  Acc: 0.8438 | Valid Loss: 0.4489  Acc: 0.8222
Epoch  5/30 | Train Loss: 0.5401  Acc: 0.8688 | Valid Loss: 0.5699  Acc: 0.8000
Epoch  6/30 | Train Loss: 0.5898  Acc: 0.8250 | Valid Loss: 0.6149  Acc: 0.6444
Epoch  7/30 | Train Loss: 0.6388  Acc: 0.8000 | Valid Loss: 0.5899  Acc: 0.7778
Epoch  8/30 | Train Loss: 0.7218  Acc: 0.8187 | Valid Loss: 0.5516  Acc: 0.7778
Epoch  9/30 | Train Loss: 0.6530  Acc: 0.8187 | Valid Loss: 0.5660  Acc: 0.7111
Epoch 10/30 | Train Loss: 0.6166  Acc: 0.8313 | Valid Loss: 0.5069  Acc: 0.7556
Epoch 11/30 | Train Loss: 0.5856  Acc: 0.8500 | Valid Loss: 0.4958  Acc: 0.8222
Epoch 12/30 | Train Loss: 0.5662  Acc: 0.8625 | Valid Loss: 0.4952  Acc: 0.7778
Epoch 13/30 | Train Loss: 0.5416  Acc: 0

In [16]:
torch.save(best_weights, "googlenet_scratch.pth")
print(f"Saved best model weights to googlenet_scratch.pth (val acc: {best_acc:.4f})")

# Final test evaluation
model_scratch.load_state_dict(best_weights)
test_loss, test_acc = evaluate(model_scratch, test_loader, device)
print(f"Test  — Loss: {test_loss:.4f}  Acc: {test_acc:.4f}")

Saved best model weights to googlenet_scratch.pth (val acc: 0.8444)
Test  — Loss: 0.3916  Acc: 0.8696


## 2. Pretrained GoogLeNet — Fine-tuned from ImageNet Weights

In [17]:
from torchvision.models import googlenet, GoogLeNet_Weights

model_pretrained = googlenet(weights=GoogLeNet_Weights.IMAGENET1K_V1, aux_logits=True)

# Replace final classifier for binary classification
num_features = model_pretrained.fc.in_features
model_pretrained.fc = nn.Linear(num_features, 1)  # single logit for BCEWithLogitsLoss

# Replace auxiliary classifiers too (they also output 1000 classes)
model_pretrained.aux1.fc2 = nn.Linear(model_pretrained.aux1.fc2.in_features, 1)
model_pretrained.aux2.fc2 = nn.Linear(model_pretrained.aux2.fc2.in_features, 1)

model_pretrained.to(device)

trainable_params = sum(p.numel() for p in model_pretrained.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable_params:,}")

Downloading: "https://download.pytorch.org/models/googlenet-1378be20.pth" to /root/.cache/torch/hub/checkpoints/googlenet-1378be20.pth


100%|██████████| 49.7M/49.7M [00:00<00:00, 158MB/s]

Trainable parameters: 9,932,963



/usr/local/lib/python3.12/dist-packages/torchvision/models/googlenet.py:341: UserWarning: auxiliary heads in the pretrained googlenet model are NOT pretrained, so make sure to train them
  warnings.warn(


In [18]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model_pretrained.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

def train_one_epoch_pretrained(model, loader, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        if model.aux_logits:
            main, aux1, aux2 = model(images)
            targets = labels.float().view(-1, 1)
            loss_main = criterion(main, targets)
            loss_aux1 = criterion(aux1, targets)
            loss_aux2 = criterion(aux2, targets)
            loss = loss_main + 0.3 * (loss_aux1 + loss_aux2)
        else:
            main = model(images)
            loss = criterion(main, labels.float().view(-1, 1))

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        probs = torch.sigmoid(main)
        preds = (probs >= 0.5).long().squeeze(1)
        correct += (preds == labels.long().to(device)).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total


def evaluate_pretrained(model, loader, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            main = model(images)
            loss = criterion(main, labels.float().view(-1, 1))

            running_loss += loss.item() * images.size(0)
            probs = torch.sigmoid(main)
            preds = (probs >= 0.5).long().squeeze(1)
            correct += (preds == labels.long().to(device)).sum().item()
            total += labels.size(0)

    return running_loss / total, correct / total

In [19]:
EPOCHS = 25
best_acc_pt = 0.0
best_weights_pt = None

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_one_epoch_pretrained(model_pretrained, train_loader, device)
    valid_loss, valid_acc = evaluate_pretrained(model_pretrained, valid_loader, device)
    scheduler.step()

    if valid_acc > best_acc_pt:
        best_acc_pt = valid_acc
        best_weights_pt = copy.deepcopy(model_pretrained.state_dict())

    print(f"Epoch {epoch:2d}/{EPOCHS} | "
          f"Train Loss: {train_loss:.4f}  Acc: {train_acc:.4f} | "
          f"Valid Loss: {valid_loss:.4f}  Acc: {valid_acc:.4f}")

print(f"\nBest validation accuracy: {best_acc_pt:.4f}")

Epoch  1/25 | Train Loss: 1.0694  Acc: 0.6000 | Valid Loss: 0.6740  Acc: 0.5111
Epoch  2/25 | Train Loss: 0.9016  Acc: 0.8313 | Valid Loss: 0.5825  Acc: 0.7111
Epoch  3/25 | Train Loss: 0.7338  Acc: 0.8938 | Valid Loss: 0.4810  Acc: 0.8444
Epoch  4/25 | Train Loss: 0.6000  Acc: 0.9125 | Valid Loss: 0.3918  Acc: 0.8889
Epoch  5/25 | Train Loss: 0.4939  Acc: 0.9313 | Valid Loss: 0.3236  Acc: 0.9111
Epoch  6/25 | Train Loss: 0.4598  Acc: 0.9313 | Valid Loss: 0.2734  Acc: 0.9333
Epoch  7/25 | Train Loss: 0.3637  Acc: 0.9875 | Valid Loss: 0.2501  Acc: 0.9333
Epoch  8/25 | Train Loss: 0.3401  Acc: 0.9875 | Valid Loss: 0.2350  Acc: 0.9333
Epoch  9/25 | Train Loss: 0.2887  Acc: 0.9875 | Valid Loss: 0.2354  Acc: 0.9333
Epoch 10/25 | Train Loss: 0.2455  Acc: 1.0000 | Valid Loss: 0.2381  Acc: 0.9333
Epoch 11/25 | Train Loss: 0.2141  Acc: 1.0000 | Valid Loss: 0.2367  Acc: 0.9333
Epoch 12/25 | Train Loss: 0.2163  Acc: 1.0000 | Valid Loss: 0.2225  Acc: 0.9556
Epoch 13/25 | Train Loss: 0.1997  Acc: 1

In [20]:
torch.save(best_weights_pt, "googlenet_pretrained.pth")
print(f"Saved best model weights to googlenet_pretrained.pth (val acc: {best_acc_pt:.4f})")

# Final test evaluation
model_pretrained.load_state_dict(best_weights_pt)
test_loss_pt, test_acc_pt = evaluate_pretrained(model_pretrained, test_loader, device)
print(f"Test  — Loss: {test_loss_pt:.4f}  Acc: {test_acc_pt:.4f}")

Saved best model weights to googlenet_pretrained.pth (val acc: 0.9556)
Test  — Loss: 0.3883  Acc: 0.8261
